# agent101 — Build Mini Claude Code from Zero

A step-by-step tutorial that teaches you how to build a mini AI coding agent from scratch.

Based on [learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) by shareAI-lab.

---

# S01: The Agent Loop

`[ s01 ] s02 > s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"One loop & PowerShell is all you need"* — one tool + one loop = an agent.
>
> **Harness layer**: The loop — the model’s first connection to the real world.

## Problem

A language model can reason about code, but it can’t *touch* the real world — can’t read files, run tests, or check errors. Without a loop, every tool call requires you to manually copy-paste results back. **You become the loop.**

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until model stops calling tools)
```

One exit condition controls the entire flow. The loop runs until the model stops calling tools.

### Step 1: Send a message to the LLM with a tool

First, let’s set up the client, define a `powershell` tool, and send a simple message to see how the LLM responds when it has a tool available.

In [1]:
import os
import json
import subprocess
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT")

SYSTEM = f"You are a coding agent at {os.getcwd()}. Use powershell to solve tasks. Act, don't explain."

TOOLS = [{
    "type": "function",
    "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}]

USER_MSG = "show current directory"
# Send a test message and inspect the raw response
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_MSG},
    ],
    tools=TOOLS,
)

msg = response.choices[0].message
print("=== Raw Response ===")
print(f"content:    {msg.content}")
print(f"tool_calls: {msg.tool_calls}")
print()
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print("=== First Tool Call ===")
    print(f"id:        {tc.id}")
    print(f"function:  {tc.function.name}")
    print(f"arguments: {tc.function.arguments}")

=== Raw Response ===
content:    None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_axVVZDg9hyaKaMMb0SsgUMVe', function=Function(arguments='{"command":"Get-Location"}', name='powershell'), type='function')]

=== First Tool Call ===
id:        call_axVVZDg9hyaKaMMb0SsgUMVe
function:  powershell
arguments: {"command":"Get-Location"}


**What happened?** The LLM didn’t reply with text — it replied with a **tool call**. It wants us to run a PowerShell command.

But nobody executed it! The model can’t touch the real world on its own. That’s what the agent loop is for — we need code that:
1. Executes the tool call
2. Sends the result back to the LLM
3. Repeats until the model stops asking for tools

### Step 2: The Tool Handler

A simple function that executes a PowerShell command and returns the output. Includes basic safety checks and a timeout.



In [2]:
def run_powershell(command: str) -> str:
    """Execute a PowerShell command with safety guards."""
    dangerous = ["Remove-Item -Recurse -Force /", "shutdown", "Restart-Computer", "Stop-Computer"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            cwd=os.getcwd(), capture_output=True, text=True, timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

# Quick test
print(run_powershell("Write-Output 'Hello from the agent!'"))

Hello from the agent!


### make real tool call
1. **User prompt** becomes the first message
2. **Send messages + tool definitions** to the LLM
3. **Check the response** — if the model didn’t call a tool, we’re done
4. **Execute each tool call**, collect results, append as messages, loop back to step 2

### Step 3: The Agent Loop


Let’s build it piece by piece.

This is the **entire secret** of an AI coding agent — a `while True` loop that:
1. Calls the LLM with the conversation history + tools
2. Appends the assistant’s response
3. If the model didn’t call any tools → **done** (exit)
4. Otherwise, execute each tool call, append results, and **loop back**

In [3]:
TOOL_HANDLERS = {"powershell": run_powershell}

def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            cmd = args.get('command', '')
            print(f"\n  [exec] {name}: {cmd}")
            output = TOOL_HANDLERS[name](**args)
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

### What just happened?

That’s the **entire agent** in under 30 lines of core logic. Let’s trace the flow:

| Step | What happens |
|------|-------------|
| User types a prompt | Added to `messages` as `{"role": "user", ...}` |
| LLM receives messages + tools | Returns either text or tool calls |
| `msg.tool_calls` is not empty | We execute each tool, append results with `role: "tool"` |
| `msg.tool_calls` is empty/None | Model is done — we exit the loop |

The `messages` list grows with each iteration, giving the LLM full context of what it has done so far.

> **Key insight**: Everything else in this course layers on top of this loop — without changing it.

## Try It!

Run the cell below to start an interactive session. Try these prompts:
1. `Create a file called hello.py that prints "Hello, World!"`
2. `List all files in the current directory`
3. `What is the current git branch?`
4. `Create a directory called test_output and write 3 files in it`

Type `q` or `exit` to quit the REPL.

In [6]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s01 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## create helloWorld.txt and add "Hello world" into it

s01 >>  q


## What Changed

| Component     | Before     | After                          |
|---------------|------------|--------------------------------|
| Agent loop    | (none)     | `while True` + tool_calls check |
| Tools         | (none)     | `powershell` (one tool)        |
| Messages      | (none)     | Accumulating list              |
| Control flow  | (none)     | Exit when `tool_calls` is empty |

---


# S02:Multiple Tools 

`s01 > [ s02 ] s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"Adding a tool means adding one handler"* — the loop stays the same; new tools register into the dispatch map.
>
> **Harness layer**: Tool dispatch — expanding what the model can reach.

The Pi agent (OpenClaw) only has **4 default tools**. Claude Code has **~ 20 default tools**. The number of tools defines what the agent can do — but the loop never changes. Adding a tool = adding one handler + one schema entry.

## Problem

With only `powershell`, the agent shells out for everything. Reading files means `Get-Content` which truncates unpredictably. Writing means `Set-Content` which can clobber files without guardrails. Every shell call is an unconstrained security surface.

Dedicated tools like `read_file` and `write_file` let you enforce **path sandboxing** at the tool level.

The key insight: **adding tools does not require changing the loop.**

## Solution

```
+--------+      +-------+      +---------------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch             |
| prompt |      |       |      | {                         |
+--------+      +---+---+      |   powershell: run_ps      |
                    ^           |   read_file:  run_read    |
                    |           |   write_file: run_write   |
                    +-----------+   edit_file:  run_edit    |
                    tool_result | }                         |
                                +---------------------------+
```

The dispatch map is a dict: `{tool_name: handler_function}`. One lookup replaces any if/elif chain.

### Step 1: Path Sandboxing

Before adding file tools, we need safety. `safe_path()` prevents the agent from escaping the workspace directory.

In [7]:
from pathlib import Path

WORKDIR = Path.cwd()

def safe_path(p: str) -> Path:
    """Resolve path and ensure it stays within the workspace."""
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

# Test: safe paths work
print(safe_path("hello.txt"))

# Test: escaping is blocked
try:
    safe_path("../../etc/passwd")
except ValueError as e:
    print(f"Blocked: {e}")

C:\Users\concao\code\agent101\hello.txt
Blocked: Path escapes workspace: ../../etc/passwd


### Step 2: New Tool Handlers

Three new handlers: `read_file`, `write_file`, `edit_file`. Each uses `safe_path()` for sandboxing.

In [8]:
def run_read(path: str, limit: int = None) -> str:
    """Read file contents, optionally limiting line count."""
    try:
        text = safe_path(path).read_text(encoding="utf-8")
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"


def run_write(path: str, content: str) -> str:
    """Write content to a file, creating parent directories if needed."""
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"


def run_edit(path: str, old_text: str, new_text: str) -> str:
    """Replace exact text in a file (first occurrence only)."""
    try:
        fp = safe_path(path)
        content = fp.read_text(encoding="utf-8")
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1), encoding="utf-8")
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

# Quick test
print(run_write("test_s02.txt", "Hello from S02!"))
print(run_read("test_s02.txt"))
print(run_edit("test_s02.txt", "S02", "Session 02"))
print(run_read("test_s02.txt"))

Wrote 15 bytes to test_s02.txt
Hello from S02!
Edited test_s02.txt
Hello from Session 02!


### Step 3: Tool Schemas & Dispatch Map

Register all 4 tools. The LLM sees the schemas; the dispatch map routes calls to handlers.

In [9]:
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "limit": {"type": "integer"},
            },
            "required": ["path"],
        },
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old_text": {"type": "string"},
                "new_text": {"type": "string"},
            },
            "required": ["path", "old_text", "new_text"],
        },
    }},
]

TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

Registered 4 tools: ['powershell', 'read_file', 'write_file', 'edit_file']


### Step 4: The Agent Loop (unchanged!)

The loop is **exactly the same** as S01. The only difference is that `TOOL_HANDLERS` now has 4 entries instead of 1.

This is the core design: the loop is generic, tools are pluggable.

In [10]:
def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            if handler:
                print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
                output = handler(**args)
            else:
                output = f"Unknown tool: {name}"
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

## Try It!

Now the agent has 4 tools. Try these prompts:
1. `Read the file pyproject.toml`
2. `Create a file called greet.py with a greet(name) function`
3. `Edit greet.py to add a docstring to the function`
4. `Read greet.py to verify the edit worked`

Watch how the LLM picks the right tool for each task — it no longer shells out for everything.

In [11]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s02 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s02 >>  reate a file called greet.py with a greet(name) function



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: reate a file called greet.py with a greet(name) function

Turn 1 - LLM Output:
  content:    {
  "command": "New-Item -Path 'C:\\Users\\concao\\code\\agent101\\greet.py' -ItemType File -Force"
}
  tool_call:  write_file({"path":"C:\\Users\\concao\\code\\agent101\\greet.py","content":"def greet(name):\n    print(f\"Hello, {name}!\")\n"})

  [exec] write_file: {"path": "C:\\Users\\concao\\code\\agent101\\greet.py", "content": "def greet(name):\n    print(f\"H
  [result] Wrote 46 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: reate a file called greet.py with a greet(name) function
  [2] assistant: (called 1 tool(s))
  [3] tool: Wrote 46 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Output

s02 >>  q


## What Changed From S01

| Component      | Before (S01)           | After (S02)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 1 (powershell only)    | 4 (powershell, read, write, edit)    |
| Dispatch       | Hardcoded handler      | `TOOL_HANDLERS` dict lookup          |
| Path safety    | None                   | `safe_path()` sandbox                |
| Agent loop     | Unchanged              | **Unchanged**                        |

> **Key insight**: Add a tool = add a handler + add a schema entry. The loop never changes.

---

**Next: Session 03 — TodoWrite** → an agent without a plan drifts. Adding a planning layer with stateful task tracking.

# S03: TodoWrite

`s01 > s02 > [ s03 ] s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"An agent without a plan drifts"* — list the steps first, then execute.
>
> **Harness layer**: Planning — keeping the model on course without scripting the route.

## Problem

On multi-step tasks, the model loses track. It repeats work, skips steps, or wanders off. Long conversations make this worse — the system prompt fades as tool results fill the context. A 10-step refactoring might complete steps 1–3, then the model starts improvising because it forgot steps 4–10.

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> | Tools   |
| prompt |      |       |      | + todo  |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                          |
              +-----------+-----------+
              | TodoManager state     |
              | [ ] task A            |
              | [>] task B  <- doing  |
              | [x] task C            |
              +-----------------------+
                          |
              if rounds_since_todo >= 3:
                inject <reminder>
```

Two mechanisms:
1. **TodoManager** — structured state the LLM writes to. Only one `in_progress` at a time (forces sequential focus).
2. **Nag reminder** — if the model goes 3+ rounds without calling `todo`, inject a nudge into the conversation.

### Step 1: TodoManager

A simple class that stores todo items with statuses: `pending`, `in_progress`, `completed`. Key constraint: **only one task can be `in_progress` at a time** — this forces the model to focus.

In [ ]:
class TodoManager:
    def __init__(self):
        self.items = []

    def update(self, items: list) -> str:
        """Validate and update the todo list."""
        if len(items) > 20:
            raise ValueError("Max 20 todos allowed")
        validated = []
        in_progress_count = 0
        for i, item in enumerate(items):
            text = str(item.get("text", "")).strip()
            status = str(item.get("status", "pending")).lower()
            item_id = str(item.get("id", str(i + 1)))
            if not text:
                raise ValueError(f"Item {item_id}: text required")
            if status not in ("pending", "in_progress", "completed"):
                raise ValueError(f"Item {item_id}: invalid status '{status}'")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item_id, "text": text, "status": status})
        if in_progress_count > 1:
            raise ValueError("Only one task can be in_progress at a time")
        self.items = validated
        return self.render()

    def render(self) -> str:
        """Display the current todo state."""
        if not self.items:
            return "No todos."
        lines = []
        for item in self.items:
            marker = {"pending": "[ ]", "in_progress": "[>]", "completed": "[x]"}[item["status"]]
            lines.append(f"{marker} #{item['id']}: {item['text']}")
        done = sum(1 for t in self.items if t["status"] == "completed")
        lines.append(f"\n({done}/{len(self.items)} completed)")
        return "\n".join(lines)


TODO = TodoManager()

# Quick test
result = TODO.update([
    {"id": "1", "text": "Read the file", "status": "completed"},
    {"id": "2", "text": "Add type hints", "status": "in_progress"},
    {"id": "3", "text": "Write tests", "status": "pending"},
])
print(result)

### Step 2: Register the `todo` Tool

The `todo` tool goes into the dispatch map like any other tool. Same pattern as S02 — add a handler + add a schema.

In [ ]:
# Reset for a fresh session
TODO = TodoManager()

# Update SYSTEM to mention todo planning
SYSTEM = f"""You are a coding agent at {WORKDIR}.
Use the todo tool to plan multi-step tasks. Mark in_progress before starting, completed when done.
Prefer tools over prose."""

# Add todo to the tool list
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    }},
    {"type": "function", "function": {
        "name": "todo",
        "description": "Update task list. Track progress on multi-step tasks. Set status to in_progress before starting, completed when done.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "string"},
                            "text": {"type": "string"},
                            "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]},
                        },
                        "required": ["id", "text", "status"],
                    },
                },
            },
            "required": ["items"],
        },
    }},
]

# Add todo handler to the dispatch map
TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":       lambda **kw: TODO.update(kw["items"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

### Step 3: Agent Loop with Nag Reminder

The loop is almost the same as S02, with one addition: a **nag reminder**.

If the model goes 3+ rounds without calling `todo`, we inject `<reminder>Update your todos.</reminder>` into the next message. This creates accountability — the model can’t just forget its plan.

For Azure OpenAI, we inject the reminder as an extra `user` message (since tool results are separate messages, not a list).

In [ ]:
def agent_loop(messages: list):
    """Agent loop with todo nag reminder."""
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Inject nag reminder if model forgot to update todos
        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder (no todo call for {rounds_since_todo} rounds)")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show LLM output
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            print(f"\n--- Final Todo State ---")
            print(TODO.render())
            return

        # Execute each tool call
        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        # Track rounds since last todo call
        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Give the agent a multi-step task and watch it plan with `todo`:
1. `Create a Python package with __init__.py, utils.py, and tests/test_utils.py`
2. `Refactor greet.py: add type hints, docstrings, and a main guard`
3. `Review all Python files in this directory and fix any style issues`

Watch for:
- The model calling `todo` to create a plan before starting
- Status transitions: `pending` → `in_progress` → `completed`
- The nag reminder if the model forgets to update its todos

In [ ]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
TODO = TodoManager()  # Reset todos for fresh session
history = []
while True:
    try:
        query = input("s03 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S02

| Component      | Before (S02)           | After (S03)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 4                      | 5 (+todo)                            |
| Planning       | None                   | TodoManager with statuses            |
| Nag injection  | None                   | `<reminder>` after 3 rounds          |
| Agent loop     | Simple dispatch        | + rounds_since_todo counter          |

> **Key insight**: The model can track its own progress — and you can see it. The "one in_progress" constraint forces sequential focus. The nag creates accountability.

---

**Next: Session 04 — Subagent** → break big tasks down; each subtask gets a clean context.